# Get Syllables

In [ ]:
import helper.library as library
import helper.syllab as syllab

In [ ]:
import time

In [ ]:
# library.py
import pandas as pd
import unicodedata

# diacritics as their unicode value
LOTONE = chr(0x0300)
HITONE = chr(0x0301)
RISETONE = chr(0x030C)
MIDTONE1 = chr(0x0304)
MIDTONE2 = chr(0x0305)
TONECHARS = {LOTONE, HITONE, RISETONE,MIDTONE1,MIDTONE2}

UNDERDOT = chr(0x0323)
UNDERLINE = chr(0x0329)
UNDERDIACS = {UNDERDOT, UNDERLINE}

# split characters into letters (with their diacritics)
def get_letters(text):
    try:
        text = unicodedata.normalize('NFD', text)
    except:
        print(text)
        return []
    text = text.replace(UNDERLINE, UNDERDOT)
    letters = []
    i = 0
    while i < len(text):
        curr_letter = ''
        # look for gb
        if ((i+1) < len(text) and ((text[i] == 'g') and (text[i+1] == 'b'))):
            curr_letter = text[i:i+2]
            i+=2

        # check if next char exists and is a diacritic
        elif ((i+1) < len(text)) and ((text[i+1] in TONECHARS) or text[i+1] in UNDERDIACS):
            if ((i+2) < len(text) and ((text[i+2] in TONECHARS) or text[i+2] in UNDERDIACS)):
                curr_letter = text[i:i+3]
                # print(f"{text[i:i+3]}\t{text[max(i-4, 0):i+4]}\t{text}")
                i+=3 # skip next two chars
            else:
                curr_letter = text[i:i+2]
                i+=2 # skip next char
                
        # normal case (the letter is one single char)
        else: 
            curr_letter = text[i]
            i+=1 # go to next char
        
        # add letter to list
        letters.append(curr_letter.lower())
    return letters

# load in data
def load_dataset(filename):
    return pd.read_csv(filename, header=0, index_col=0)

In [ ]:
# syllab.py
from enum import Enum
from helper import library

# define the possible types of letters
class Letters(Enum):
    SP = 0  # space
    C = 1   # consonant
    M = 2   # m (could be syllabic or consonant)
    N = 3   # n (could be syllabic, consonant, or nasal vowel)
    V = 4   # vowel

# helper functions
def ischar(text):
    return text[0].isalpha()

def chartype(text):
    if ischar(text): 
        if text[0] in ['a', 'e', 'i', 'o', 'u']: return Letters.V
        elif text[0] == 'm': return Letters.M
        elif text[0] == 'n': return Letters.N
        else: return Letters.C
    else: return Letters.SP # punctuation counts as spacing

# syllabify a list of letters, assuming len(letters) != 0
def get_next_syll(letters, syllables, print_it=False):    
    # get types of letters
    # get last char's type
    types = [chartype(letters[-1])]
    
    # get second and third to last chars (if present)
    if len(letters) > 2:
        types.insert(0, chartype(letters[-2]))
        types.insert(0, chartype(letters[-3]))
    # get second to last char, default third to last to SP
    elif len(letters) > 1: 
        types.insert(0, chartype(letters[-2]))
        types.insert(0, Letters.SP)
    # set other values to default value SP
    else:
        types.insert(0, Letters.SP)
        types.insert(0, Letters.SP)
    
    if print_it: print('Types:', types)

    # identify the syllable
    # logic based on Kumolalo 2010
    curr_syll = [] # store current identified syllable
    # handle when last thing isn't a letter
    if (types[-1] == Letters.SP):
        curr_syll = ['SP']
        letters = letters[:-1]
        if print_it: print('SP curr_syll : ', curr_syll)
    # look for CVn (same as DVn in this set up)
    elif (types[-3] in [Letters.C, Letters.M, Letters.N]) and (types[-2] == Letters.V) and (types[-1] == Letters.N):
        curr_syll = letters[-3:]
        letters = letters[:-3]
        if print_it: print('CVn curr_syll : ', curr_syll)
    # look for CV (same as DV)
    elif (types[-2] in [Letters.C, Letters.M, Letters.N]) and (types[-1] == Letters.V):
        curr_syll = letters[-2:]
        letters = letters[:-2]
        if print_it: print('CV curr_syll : ', curr_syll)
    # look for Vn
    elif (types[-2] == Letters.V) and (types[-1] == Letters.N):
        curr_syll = letters[-2:]
        letters = letters[:-2]
        if print_it: print('Vn curr_syll : ', curr_syll)
    # look for V
    elif types[-1] == Letters.V:
        curr_syll = letters[-1:]
        letters = letters[:-1]
        if print_it: print('V curr_syll : ', curr_syll)
    # look for N
    elif types[-1] in [Letters.N, Letters.M]:
        curr_syll = letters[-1:]
        letters = letters[:-1]
        if print_it: print('N curr_syll : ', curr_syll)
    # handle other scenario (must be an error)
    else:
        curr_syll = ['ERR']
        letters = letters[:-1]
        if print_it: print('ERR : ', letters[-3:])

    syllables.insert(0,curr_syll) # add syllable to front of list
    return letters, syllables

def syllabify_letters(letters, print_it = False):
    syllables = []
    while (len(letters) > 0):
        if print_it: print('get_next_syll')
        letters, syllables = get_next_syll(letters, syllables)

    return syllables

def syllabify_df(df):
    syllables = []
    for id, row in df.iterrows():
        letters = library.get_letters(row['sentence'])
        curr_sylls = syllabify_letters(letters)
        syllables.append([id, curr_sylls])
    return syllables

In [ ]:
full_train = pd.read_csv('/Users/graven2/Documents/THESIS/data/targets-train-val.txt', names=['sentence'])
syllables = syllabify_df(full_train)
all_syllables = pd.DataFrame(columns=['Source', 'ID', 'Sentence', 'Syllables'])
for row in syllables:
    source = 'Unidentifed'
    id = row[0]
    sentence = full_train.loc[id, 'sentence']
    syllables = row[1]
    all_syllables.loc[len(all_syllables)] = [source, id, sentence, syllables]


In [ ]:
print(all_syllables)
# all_syllables.to_csv('syllabified.csv')

In [ ]:
# Get syllable and word counts
def dup_rows(df):
    print(f'Original Length: {len(df)}')
    dup_sentences = df.duplicated(subset='Sentence', keep=False)
    dup_indices = dup_sentences[dup_sentences == True]
    print(f'{len(dup_indices)} TOTAL DUPLICATES FOUND')
    for i in dup_indices.index:
        print(f'{i}: {df.loc[i]}')
    dropped = df.drop_duplicates(subset='Sentence',keep='first')
    print(f'New Length: {len(dropped)}')
    return dropped

deduped = dup_rows(all_syllables)

In [ ]:
import string

deduped = deduped.copy()
all_words = set()
def count_syls(row):
    count = 0
    for syl in row['Syllables']:
        if syl[0] != 'SP': count+=1
    return count

def count_words(row, all_words):
    count = 0
    try: 
        text = row['Sentence']
        words = text.split()
        words = [s.translate(str.maketrans('', '', string.punctuation)) for s in words]
        # print(words)
        all_words.update(words)
        return len(words)
    except: return 0


test_row = deduped.iloc[0]
text = test_row['Sentence']
words = text.split()
no_punc = [s.translate(str.maketrans('', '', string.punctuation)) for s in words]
print(no_punc)


deduped['Syl Count'] = deduped.apply(lambda row: count_syls(row), axis =1)
deduped['Word Count'] = deduped.apply(lambda row: count_words(row, all_words), axis=1)

# for row in deduped.iterrows():
#     syllable_rows = deduped['Syllables']
#     for syllables in syllable_rows:
#         for syl in syllables:
#             # print(syl)
#             if syl[0] != 'SP': total_syls+=1

print(deduped['Syl Count'].sum())
print(deduped['Word Count'].sum())

# print(deduped)

print(len(all_words))

# Run N-Grams

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [ ]:
# ngrams.py
from helper import library
from enum import Enum
from helper import syllab

# Create consistent labels for tones
class Tones(Enum):
    H = 1
    M = 0
    L = -1

# possible vowels in Yoruba
VOWELS = ['a','e','i','o','u']

# remove the diacritics from a syllable
# can keep 'underdiacs', 'tone', or 'none'
def _rm_diacritics_syll(syllable, keep='underdiacs'):
    new_syll = []
    for letter in syllable:
        # corner cases
        if letter == 'SP': return syllable
        if letter == 'ERR': return syllable

        # normal syllable
        include = [letter[0]] # keep original letter
        
        # check second char
        if (len(letter) > 1) and ((letter[1] in library.UNDERDIACS) and (keep=='underdiacs')):
            include.append(letter[1])
        if (len(letter) > 1) and ((letter[1] in library.TONECHARS) and (keep=='tone')):
            include.append(letter[1])

        # check third char
        if (len(letter) > 2) and ((letter[2] in library.UNDERDIACS) and (keep=='underdiacs')):
            include.append(letter[2])
        if (len(letter) > 2) and ((letter[2] in library.TONECHARS) and (keep=='tone')):
            include.append(letter[2])

        # create string from included characters
        new_syll.append(''.join(include))
    return new_syll

# remove diacritics from a set of syllables
def rm_diacritics_row(row, keep='underdiacs'):
    return [_rm_diacritics_syll(x, keep) for x in row['Syllables']]

# remove diacritics from df
# can keep 'underdiacs', 'tone', or 'none'
def rm_diacritics_df(df, keep='underdiacs'):
    new_df = df.copy()
    new_df['Syllables'] = new_df.apply(lambda row: rm_diacritics_row(row, keep), axis=1)
    return new_df

# return the index where the tone carrier is located in syllable
def _tone_carrier_index(syllable):
    # find tone carrier in syllable (either N or M alone, or V)
    if len(syllable) == 1: return 0
    
    # if len > 0, tone carrier MUST be a vowel; at most 1 vowel per syll
    else: 
        if (syllable[0][0] in VOWELS): return 0
        else: return 1 # vowel MUST be first or second in syllable

# identify the tone of a syllable
# returns 
def get_tone(syllable):
    # corner cases
    if syllable[0] == 'SP' or syllable[0] == 'ERR': return Tones.M

    tone_carrier = syllable[_tone_carrier_index(syllable)]

    # get tone from tone carrier
    if (len(tone_carrier) > 1) and (tone_carrier[1] in library.TONECHARS):
        tone = tone_carrier[1]
    elif (len(tone_carrier) > 2) and (tone_carrier[2] in library.TONECHARS):
        tone = tone_carrier[2]
    else: return Tones.M # no tone present --> Mid tone

    if tone == library.LOTONE: return Tones.L
    elif tone == library.HITONE: return Tones.H
    else: return Tones.M # default/error is midtone

# add a given tone to a syllable
def _add_tone(syllable, tone):
    if tone == Tones.H: tone_char = library.HITONE
    elif tone == Tones.L: tone_char = library.LOTONE
    else: return syllable # mid tone is unmarked

    index = _tone_carrier_index(syllable)
    new_syll = syllable[:]
    new_syll[index] = ''.join([syllable[index], tone_char])
    return new_syll

# find the n-sized context string for syllable i in syllables
def _get_context(syllables, n, i):
    context = []
    for j in range(1, n+1):
        # get -Syl
        # insert at FRONT of list
        if (i-j >= 0): 
            curr_context = syllables[i-j]
            curr_context = _rm_diacritics_syll(curr_context)
            curr_context_str = ''.join(curr_context)
            context.insert(0, curr_context_str)
        else: context.insert(0, '<') # start of sentence token

        # get +Syl
        # add to END of list
        if (i+j < len(syllables)):
            curr_context = syllables[i+j]
            curr_context = _rm_diacritics_syll(curr_context)
            curr_context_str = ''.join(curr_context)
            context.append(curr_context_str)
        else: context.append('>') # end of sentence token
    
    # merge context into a string
    context_str = '.'.join(context) # -Syl.-Syl.+Syl.+Syl
    return context_str


# get n-grams
# n is the number of items before and after to consider
def _syll_grams(syllables, counts, n):
    # counts has format {syll: {-Syl.-Syl.+Syl.+Syl : {H : count, M : count, L : count}}}
    for i, syll in enumerate(syllables):
        syll_str = ''.join(_rm_diacritics_syll(syll))

        # get contexts for this syllable so far
        poss_contexts = counts.get(syll_str, dict())

        # get current context for syllable
        context_str = _get_context(syllables, n, i)

        # update with new tones
        context_tones = poss_contexts.get(context_str, dict())
        curr_tone = get_tone(syll)
        curr_tone_count = context_tones.get(curr_tone, 0)

        # update all dictionaries
        context_tones.update({curr_tone : curr_tone_count + 1})
        poss_contexts.update({context_str : context_tones})
        counts.update({syll_str : poss_contexts})

    return counts

# create a full n-gram count from a df
def create_syll_grams(df, n=4):
    counts = dict()
    for _, row in df.iterrows():
        counts = _syll_grams(row['Syllables'], counts, n)
    return counts

# predict the tone for each syllable in list of syllables
def pred_tone(syllables, counts, n=4):
    with_tones = []
    
    for i, syll in enumerate(syllables):
        # corner case
        if syll[0] == 'SP' or syll[0] == 'ERR':
            with_tones.append(syll)
            continue

        # get current syllable and its context
        syll_str = ''.join(syll)
        context_str = _get_context(syllables, n, i)

        # collect stored counts
        poss_tones = counts.get(syll_str, dict()).get(context_str, dict())

        # get most frequent tone
        h = poss_tones.get(Tones.H, 0)
        m = poss_tones.get(Tones.M, 0)
        l = poss_tones.get(Tones.L, 0)
        if (h > m) and (h > l): new_syll = _add_tone(syll, Tones.H)
        elif (l > m) and (l > h): new_syll = _add_tone(syll, Tones.L)
        else: new_syll = _add_tone(syll, Tones.M)
        with_tones.append(new_syll)

    return with_tones

# do full predictions
def predict_all_tones(df, counts, n=4):
    new_df = df.copy()
    new_df['Prediction'] = new_df.apply(lambda row: pred_tone(row['Syllables'], counts, n), axis=1)
    return new_df

# calculate word error rate for a row, returns (wrong words, total words)
def _eval_row(row, print_it=False):
    correct = row['Syllables']
    pred = row['Prediction']

    wrong_words = 0
    total_words = 0
    in_word = False # identifies whether currently in a word or not
    curr_word_accurate = True # identifies whether the current word has gotten a tone wrong yet

    # iterate through syllables
    for i in range(len(correct)):
        # check if tones match
        corr_tone = get_tone(correct[i])
        pred_tone = get_tone(pred[i])
        if corr_tone != pred_tone: 
            curr_word_accurate = False

        # check if a word is finished
        if in_word:
            # word has ended
            if correct[i][0] == 'SP':
                in_word = False
                if not curr_word_accurate: 
                    wrong_words += 1
                    if print_it: print('WRONG', correct,pred)
                total_words += 1
                curr_word_accurate = True # reset accuracy
        if correct[i][0] != 'SP': in_word = True
    # check corner case: word never ended yet
    if in_word:
        if not curr_word_accurate: wrong_words+=1
        total_words+=1

    return pd.Series({'Wrong Words' : wrong_words, 'Total Words' : total_words})

# determine wrong words in df of syllables
def evaluate(df, print_it=False):
    new_df = df.copy()
    new_df[['Wrong Words', 'Total Words']] = new_df.apply(lambda row: _eval_row(row, print_it=print_it), axis=1, result_type='expand')
    return new_df

#


In [ ]:
tester = all_syllables.loc[0, 'Syllables']
print(tester)

# 29 = ['r', 'ọ̀']
# 2 = ['SP']
# 6 = ['í']
# 22 = ['t', 'u', 'n']
testI = 22
orig_syll = tester[testI]
new_syll = _rm_diacritics_syll(orig_syll)
print(orig_syll, new_syll)

print(get_tone(orig_syll))
print(get_tone(new_syll))

minitester = tester[0:30]
ngram_counts = _syll_grams(minitester, dict(), 4)
print(ngram_counts)

untoned = [_rm_diacritics_syll(x) for x in minitester]
print(untoned)
toned = pred_tone(untoned, ngram_counts, 4)
print(toned)

In [ ]:
tester = all_syllables.loc[0:5].copy()
counts = create_syll_grams(tester)
print(counts)

detoned = tester.apply(lambda row: rm_diacritics_row(row), axis=1)
tester['Prediction'] = detoned.apply(lambda row: pred_tone(row, counts))

testI = 4
print(tester.loc[testI, 'Syllables'])
print(tester.loc[testI, 'Prediction'])

tester = evaluate(tester)
testI = 4
print(tester.loc[testI, 'Total Words'], tester.loc[testI, 'Sentence'])
wer = (tester['Wrong Words'].sum() / tester['Total Words'].sum()) * 100
print(wer)

In [ ]:
def dup_rows(df):
    print(f'Original Length: {len(df)}')
    dup_sentences = df.duplicated(subset='Sentence', keep=False)
    dup_indices = dup_sentences[dup_sentences == True]
    print(f'{len(dup_indices)} TOTAL DUPLICATES FOUND')
    for i in dup_indices.index:
        print(f'{i}: {df.loc[i]}')
    dropped = df.drop_duplicates(subset='Sentence',keep='first')
    print(f'New Length: {len(dropped)}')
    return dropped

deduped = dup_rows(all_syllables)

In [ ]:
# split data
curr_df = deduped
# split into 10 folds, make it even across all three datasets
skf = StratifiedKFold(n_splits=10,shuffle=False)
X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
y = curr_df['Source'].to_numpy() # stratify across the datasets

for n in range(0, 5):
    print(f'#########\n#n = {n}\n#########')
    # run predictions for all folds
    WERs = []
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        # if i > 1: break
        # n = 0

        print(f"--- Fold {i} ---")
        train_df = curr_df.iloc[train_index].copy()
        test_df = curr_df.iloc[test_index].copy()

        # create n-grams
        start_time = time.time()
        counts = create_syll_grams(train_df, n=n)
        end_time = time.time()
        print('N-Gram Timing : ', (end_time-start_time), flush=True)

        # create detoned version (to test predictions)
        start_time = time.time()
        detoned = test_df.apply(lambda row: rm_diacritics_row(row), axis=1)

        # predict tones
        test_df['Prediction'] = detoned.apply(lambda row: pred_tone(row, counts, n=n))
        end_time = time.time()
        print('Prediction Timing : ', (end_time - start_time), flush=True)

        # evaluate
        start_time = time.time()
        test_df = evaluate(test_df, print_it=False)
        wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
        end_time = time.time()
        print('Evaluation Timing : ', (end_time - start_time), flush=True)
        print('WER = ', wer, flush=True)
        WERs.append(wer)
        # if i ==0:
        #     display(test_df.sample(10))

        # print(f"  Train: index={train_index}")
        # print(f"  Test:  index={test_index}")

    print(float(sum(WERs)) / float(len(WERs)))

In [ ]:
print(float(sum(WERs)) / float(len(WERs)))